<a href="https://colab.research.google.com/github/Apekshagaonkar/ai-systems-notebooks/blob/main/week1_production_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 — Production Python
**A notebook proves your code worked once. Production code has to work every time, for anyone.**

| # | Tool | The question it answers |
|---|---|---|
| 1 | Classes | What is this thing, and what can it do? |
| 2 | Type hints & docstrings | What goes in, what comes out, what does it mean? |
| 3 | Dataclasses | Do I have to write all this by hand? |
| 4 | Pydantic | Can I trust this data? |
| 5 | pytest | Does it still work after I change something? |
| 6 | Packages | How do I use it from somewhere else? |

### Setup — run this first

In [ ]:
!pip -q install pydantic pytest
import pydantic, pytest
print("pydantic", pydantic.VERSION, "| pytest", pytest.__version__)

pydantic 2.13.5 | pytest 8.4.2


## 1. Classes — keep data and its rules together

Let's start with something that looks completely harmless.

### 💥 The problem: a dictionary has no rules

In [ ]:
reading = {"location": "New York", "temperature_c": 21.5}

# A typo when updating...
reading["temprature_c"] = 25.0

print(reading)

{'location': 'New York', 'temperature_c': 21.5, 'temprature_c': 25.0}


### ✅ The fix: a class bundles data (attributes) with behavior (methods)

In [ ]:
class Thermometer:
    def __init__(self, location):
        # __init__ runs automatically when the object is created
        self.location = location   # self = "this particular thermometer"
        self.readings = []

    def add(self, reading):
        self.readings.append(reading)


t = Thermometer("New York")
t.add(21.5)
t.add(23.0)
print(t)   # what does printing the object show?

In [ ]:
class Thermometer:
    def __init__(self, location):
        self.location = location
        self.readings = []

    def add(self, reading):
        self.readings.append(reading)

    def __repr__(self):
        return f"Thermometer(location={self.location!r}, readings={self.readings})"


t = Thermometer("New York")
t.add(21.5)
print(t)

Thermometer(location='New York', readings=[21.5])


### 🔁 Pattern: configure once, apply many times
Set the rules once in `__init__`, then apply them to many records.

In [ ]:
class RangeChecker:
    def __init__(self, min_c, max_c):
        self.min_c = min_c
        self.max_c = max_c

    def is_valid(self, temp_c):
        return self.min_c <= temp_c <= self.max_c


checker = RangeChecker(min_c=-10, max_c=45)     # configure ONCE
for temp in [21.5, 60.0, -3.0]:                  # apply MANY times
    print(temp, "->", checker.is_valid(temp))

21.5 -> True
60.0 -> False
-3.0 -> True


## 2. Type hints & docstrings


### ✅ Type hints put the contract in the signature

In [ ]:
def average(temps: list[float]) -> float:
    return sum(temps) / len(temps)

def find_station(station_id: str) -> str | None:   # None = "might not find it"
    return None

print(average([20.0, 22.0, 24.0]))

22.0


 Python does NOT enforce them

In [ ]:
def double(x: int) -> int:
    return x * 2

print(double(4))
print(double("ab"))   # we promised an int...

8
abab


### 📝 Docstrings: hints say *what type*, docstrings say *what it means*

In [ ]:
def average(temps: list[float]) -> float:
    """Return the mean temperature.

    Temperatures are in degrees Celsius.
    Raises ZeroDivisionError if temps is empty.
    """
    return sum(temps) / len(temps)

help(average)

Help on function average in module __main__:

average(temps: list[float]) -> float
    Return the mean temperature.

    Temperatures are in degrees Celsius.
    Raises ZeroDivisionError if temps is empty.



## 3. Dataclasses — remove the boilerplate

### 💥 The problem: every field written again and again

In [ ]:
class HourlyReadingManual:
    def __init__(self, timestamp: str, temperature_c: float, precipitation_mm: float):
        self.timestamp = timestamp
        self.temperature_c = temperature_c
        self.precipitation_mm = precipitation_mm

    def __repr__(self):
        return (f"HourlyReadingManual(timestamp={self.timestamp!r}, "
                f"temperature_c={self.temperature_c}, precipitation_mm={self.precipitation_mm})")

    def __eq__(self, other):
        return (self.timestamp, self.temperature_c, self.precipitation_mm) == \
               (other.timestamp, other.temperature_c, other.precipitation_mm)

print(HourlyReadingManual("2026-04-08T00:00", 20.1, 0.0))

HourlyReadingManual(timestamp='2026-04-08T00:00', temperature_c=20.1, precipitation_mm=0.0)


### ✅ The fix: `@dataclass` writes `__init__`, `__repr__`, `__eq__` for you

In [ ]:
from dataclasses import dataclass, field

@dataclass
class HourlyReading:
    timestamp: str
    temperature_c: float
    precipitation_mm: float


a = HourlyReading("2026-04-08T00:00", 20.1, 0.0)
b = HourlyReading("2026-04-08T00:00", 20.1, 0.0)
print(a)
print("equal?", a == b)

HourlyReading(timestamp='2026-04-08T00:00', temperature_c=20.1, precipitation_mm=0.0)
equal? True


### ⚠️ Rule 1: mutable defaults

First, the underlying Python trap — in an ordinary class, a default list is created **once** and shared:

In [ ]:
class Station:
    def __init__(self, name, readings=[]):   # the trap
        self.name = name
        self.readings = readings

s1 = Station("A")
s2 = Station("B")
s1.readings.append(21.5)
print("s2 readings:", s2.readings)   # we never touched s2!

s2 readings: [21.5]


In [ ]:
try:
    @dataclass
    class BadStation:
        name: str
        readings: list[float] = []
except ValueError as e:
    print("ValueError:", e)


@dataclass
class GoodStation:
    name: str
    readings: list[float] = field(default_factory=list)   # fresh list per object

g1, g2 = GoodStation("A"), GoodStation("B")
g1.readings.append(21.5)
print(g1, g2)

ValueError: mutable default <class 'list'> for field readings is not allowed: use default_factory
GoodStation(name='A', readings=[21.5]) GoodStation(name='B', readings=[])


### 🔒 Rule 2: `frozen=True` = read-only (and hashable)

In [ ]:
@dataclass(frozen=True)
class Location:
    lat: float
    lon: float

home = Location(18.52, 73.86)
try:
    home.lat = 0.0
except Exception as e:
    print(type(e).__name__, "->", e)

print("usable as a dict key:", {home: "New York"})

FrozenInstanceError -> cannot assign to field 'lat'
usable as a dict key: {Location(lat=18.52, lon=73.86): 'New York'}


## 4. Pydantic — validate at the boundary

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Reading(BaseModel):
    station_id: str = Field(min_length=3)
    temperature_c: float = Field(ge=0, le=60)     # ge = >=, le = <=
    humidity: float = Field(ge=0, le=100)

### ✅ Good outside data — note the strings

In [ ]:
r = Reading(station_id="NYC01", temperature_c="21.5", humidity="60")
print(r)
print(type(r.temperature_c))

station_id='NYC01' temperature_c=21.5 humidity=60.0
<class 'float'>


### 💥 Bad outside data — three problems at once

In [ ]:
try:
    Reading(station_id="AB", temperature_c=150, humidity="very humid")
except ValidationError as e:
    print(e)

3 validation errors for Reading
station_id
  String should have at least 3 characters [type=string_too_short, input_value='AB', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
temperature_c
  Input should be less than or equal to 60 [type=less_than_equal, input_value=150, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
humidity
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='very humid', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


### Rules across two fields: `model_validator`

In [ ]:
from pydantic import model_validator

class DailySummary(BaseModel):
    min_c: float
    max_c: float

    @model_validator(mode="after")
    def check_order(self):
        if self.min_c > self.max_c:
            raise ValueError("min_c must not exceed max_c")
        return self

try:
    DailySummary(min_c=30, max_c=10)
except ValidationError as e:
    print(e)

1 validation error for DailySummary
  Value error, min_c must not exceed max_c [type=value_error, input_value={'min_c': 30, 'max_c': 10}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


### 🧭 Which one should I use?

| Data came from... | Use |
|---|---|
| Outside (API, file, form) — untrusted | **Pydantic model** |
| Inside (our own code) — trusted | **Dataclass** |

## 5. pytest — write the check down


### 💥 Why floats need special care

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(0.1 + 0.2 == pytest.approx(0.3))

0.30000000000000004
False
True


### 📦 Build a small package: `weatherkit`
Real tests live in files, so we create files from the notebook with `%%writefile`.

```
weatherkit/__init__.py   ← public entry point
weatherkit/models.py     ← our dataclass + logic
tests/test_models.py     ← the tests
conftest.py              ← empty; lets tests import weatherkit
```

In [ ]:
import os
os.makedirs("weatherkit", exist_ok=True)
os.makedirs("tests", exist_ok=True)
open("conftest.py", "w").close()   # empty file at the project root

In [ ]:
%%writefile weatherkit/models.py
from dataclasses import dataclass


@dataclass
class HourlyReading:
    """One hourly weather observation. Temperature in Celsius, rain in mm."""
    timestamp: str
    temperature_c: float
    precipitation_mm: float


def to_fahrenheit(celsius: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32

Writing weatherkit/models.py


In [ ]:
%%writefile weatherkit/__init__.py
from weatherkit.models import HourlyReading, to_fahrenheit

Writing weatherkit/__init__.py


In [ ]:
%%writefile tests/test_models.py
import pytest
from weatherkit import HourlyReading, to_fahrenheit


def test_reading_stores_temperature():
    day = HourlyReading("2026-04-08T00:00", 20.1, 0.0)
    assert day.temperature_c == pytest.approx(20.1)


@pytest.mark.parametrize("c, f", [(0, 32), (100, 212), (-40, -40), (37, 98.6)])
def test_to_fahrenheit(c, f):
    assert to_fahrenheit(c) == pytest.approx(f)


def test_bad_input_fails():
    with pytest.raises(TypeError):
        to_fahrenheit("hot")

Writing tests/test_models.py


In [ ]:
!python -m pytest -q

......                                                                   [100%]
6 passed in 0.03s


### 🔨 A test that cannot fail is useless — break the code on purpose

In [ ]:
%%writefile weatherkit/models.py
from dataclasses import dataclass


@dataclass
class HourlyReading:
    """One hourly weather observation. Temperature in Celsius, rain in mm."""
    timestamp: str
    temperature_c: float
    precipitation_mm: float


def to_fahrenheit(celsius: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 23    # BUG: typed 23 instead of 32

Overwriting weatherkit/models.py


In [ ]:
!python -m pytest -q

.FFFF.                                                                   [100%]
=================================== FAILURES ===================================
___________________________ test_to_fahrenheit[0-32] ___________________________

c = 0, f = 32

    @pytest.mark.parametrize("c, f", [(0, 32), (100, 212), (-40, -40), (37, 98.6)])
    def test_to_fahrenheit(c, f):
>       assert to_fahrenheit(c) == pytest.approx(f)
E       assert 23.0 == 32 ± 3.2e-05
E         
E         comparison failed
E         Obtained: 23.0
E         Expected: 32 ± 3.2e-05

tests/test_models.py:12: AssertionError
_________________________ test_to_fahrenheit[100-212] __________________________

c = 100, f = 212

    @pytest.mark.parametrize("c, f", [(0, 32), (100, 212), (-40, -40), (37, 98.6)])
    def test_to_fahrenheit(c, f):
>       assert to_fahrenheit(c) == pytest.approx(f)
E       assert 203.0 == 212 ± 2.1e-04
E         
E         comparison failed
E         Obtained: 203.0
E         Expected: 212 ±

In [ ]:
%%writefile weatherkit/models.py
from dataclasses import dataclass


@dataclass
class HourlyReading:
    """One hourly weather observation. Temperature in Celsius, rain in mm."""
    timestamp: str
    temperature_c: float
    precipitation_mm: float


def to_fahrenheit(celsius: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32

Overwriting weatherkit/models.py


In [ ]:
!python -m pytest -q

......                                                                   [100%]
6 passed in 0.03s


## 6. Scripts vs. libraries & wrap-up

### 💥 Importing a file runs all its top-level code

In [ ]:
%%writefile noisy.py
def greet():
    return "hello"

print("Downloading 2 GB of weather data...")   # top-level work!

Writing noisy.py


In [ ]:
import noisy   # we only wanted greet()...

### ✅ The fix: put the work in `main()` behind the `__main__` guard

In [ ]:
%%writefile quiet.py
def greet():
    return "hello"


def main():
    print("Downloading 2 GB of weather data...")


if __name__ == "__main__":   # True only when run directly
    main()

Writing quiet.py


In [ ]:
import quiet          # imports silently
print(quiet.greet())

hello


In [ ]:
!python quiet.py

---

## 🔭 Where this leads: Week 4

```python
class WeatherClassifier:
    def __init__(self, model_path: str):
        self.model = load(model_path)          # configure ONCE

    def predict(self, reading: HourlyReading) -> str:
        return self.model.predict(...)         # apply MANY times
```

Same class shape, same package layout, same tests.

## ✅ Recap

1. **Classes** keep data and behavior together.
2. **Type hints & docstrings** document the contract — but hints aren't enforced.
3. **Dataclasses** remove boilerplate for data-holding classes.
4. **Pydantic** enforces types and rules at the boundary.
5. **pytest** saves your checks; **packages** make code importable.

> - Data from a CSV: dataclass or Pydantic model? Why?
> - If type hints aren't enforced, why write them?
> - Your test passes. How do you know it's testing anything?